# Week 92, Day 1: Release

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w092d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('pyproject.toml', 'w', encoding='utf-8') as f:
    f.write('[build-system]\nrequires = ["setuptools>=69"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "waymark-walks"\nversion = "1.0.0"\ndescription = "Plan walking routes with honest times."\nreadme = "README.md"\nrequires-python = ">=3.11"\nlicense = "MIT"\nauthors = [{name = "A. Walker"}]\ndependencies = ["matplotlib>=3.8", "pandas"]\n\n[project.scripts]\nwaymark = "waymark.cli:main"\n')

**Predict before you run:** what will this print?



In [ ]:
tags = ["v0.9.0", "v0.10.0", "v0.2.1"]
print(max(tags))
print(max(tags, key=lambda t: tuple(int(p) for p in t[1:].split("."))))

> **Key idea.** Version with semver: PATCH for fixes, MINOR for compatible features (PATCH to 0), MAJOR for breaking changes (MINOR and PATCH to 0); compare versions as tuples of numbers. Conventional commits (`feat`, `fix`, `!` or `BREAKING CHANGE`) give the next version and the notes. Describe the package fully in `pyproject.toml`, with bounded dependencies and a script. Build an sdist and a wheel, try TestPyPI first, tag to match, and publish SHA-256 checksums.

### Exercise: Versions and bumps

Write `parse_version(text)`: accept `MAJOR.MINOR.PATCH`, perhaps with a leading `v`, each part a whole number without leading zeros (except `0` itself), and return a tuple of ints; otherwise raise `ValueError("not a version: TEXT")`. Then `bump(version, part)` returning the next version string for `"major"`, `"minor"` or `"patch"` (raising `ValueError` for anything else).

In [ ]:
import re

def parse_version(text):
    ___

def bump(version, part):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_parse():\n    \"numbers, not text\"\n    assert parse_version(\"1.2.3\") == (1, 2, 3) and parse_version(\"v0.10.0\") == (0, 10, 0)\n    assert parse_version(\"v0.10.0\") > parse_version(\"v0.9.0\")\n    for bad in [\"1.2\", \"1.2.3.4\", \"01.2.3\", \"1.2.x\", \"\", \"version 1.2.3\", \"1.2.3 \"]:\n        try:\n            parse_version(bad)\n        except ValueError as e:\n            assert str(e) == f\"not a version: {bad}\"\n        else:\n            raise AssertionError(repr(bad))\n\ndef test_bump():\n    \"the parts reset\"\n    assert bump(\"1.2.3\", \"patch\") == \"1.2.4\" and bump(\"1.2.3\", \"minor\") == \"1.3.0\" and bump(\"1.2.3\", \"major\") == \"2.0.0\"\n    assert bump(\"v0.9.9\", \"minor\") == \"0.10.0\"\n    try:\n        bump(\"1.2.3\", \"micro\")\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"micro\")")

### Exercise: The next version from the commits

Write `next_version(current, commits)`: each commit is a message whose first line may follow `TYPE(SCOPE)!: DESCRIPTION` (scope and `!` optional). A commit is **breaking** if it has the `!` or any line of it starts with `BREAKING CHANGE:`; otherwise `feat` asks for a minor release and `fix` or `perf` a patch; other types and other messages ask for none. Return the next version for the largest change asked for, or `None` if none is. Below `1.0.0`, a breaking change bumps the **minor** version.

In [ ]:
import re

def next_version(current, commits):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_next():\n    \"the largest change wins\"\n    assert next_version(\"1.0.0\", [\"fix: round walks ignored the target time\", \"docs: how to share\"]) == \"1.0.1\"\n    assert next_version(\"1.0.1\", [\"fix(map): Island\", \"feat(share): open a route as GeoJSON\", \"test: more\"]) == \"1.1.0\"\n    assert next_version(\"1.1.0\", [\"feat(api)!: rename /plan to /v1/routes\"]) == \"2.0.0\"\n    assert next_version(\"1.1.0\", [\"refactor: tidy\\n\\nBREAKING CHANGE: Route.minutes is now a method\"]) == \"2.0.0\"\n    assert next_version(\"1.1.0\", [\"docs: README\", \"chore: tidy\", \"Merge branch 'main'\"]) is None\n    assert next_version(\"0.4.2\", [\"feat!: new map format\"]) == \"0.5.0\" and next_version(\"0.4.2\", [\"perf: faster A*\"]) == \"0.4.3\"")

### Exercise: Release notes from the commits

Write `release_notes(version, date, commits)`: start with `## [VERSION] - DATE`; then, for each of `Breaking`, `Added` and `Fixed` that has entries, a blank line, `### SECTION` and one line `- DESCRIPTION` per commit (with ` (SCOPE)` after it if there's a scope), in commit order. A breaking commit (`!` in the header, or a `BREAKING CHANGE:` line) goes only under Breaking; `feat` under Added; `fix` and `perf` under Fixed; others are left out. Return the lines joined with newlines.

In [ ]:
import re

def release_notes(version, date, commits):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("COMMITS = [\"feat(share): open a shared route as GeoJSON\", \"fix: round walks ignored the target time\", \"docs: how to share\",\n           \"feat(api)!: rename /plan to /v1/routes\", \"perf(search): A* with a grid index\", \"feat: statistics page\"]\n\ndef test_notes():\n    \"for users, grouped\"\n    assert release_notes(\"2.0.0\", \"2026-11-16\", COMMITS) == (\n        \"## [2.0.0] - 2026-11-16\\n\\n### Breaking\\n- rename /plan to /v1/routes (api)\\n\\n\"\n        \"### Added\\n- open a shared route as GeoJSON (share)\\n- statistics page\\n\\n\"\n        \"### Fixed\\n- round walks ignored the target time\\n- A* with a grid index (search)\")\n    assert release_notes(\"1.0.1\", \"2026-11-17\", [\"fix: Island\"]) == \"## [1.0.1] - 2026-11-17\\n\\n### Fixed\\n- Island\"\n    assert release_notes(\"1.0.1\", \"2026-11-17\", [\"chore: tidy\"]) == \"## [1.0.1] - 2026-11-17\"")

### Exercise: Checking pyproject.toml

Write `pyproject_problems(text)` with `tomllib`. Return `["not valid TOML"]` if it doesn't parse, or `["no [project] table"]` if there's no `project`. Otherwise, in this order: `"missing project.KEY"` for each of `name`, `version`, `description`, `readme`, `requires-python` and `license` that's missing; `"version V is not MAJOR.MINOR.PATCH"` if the version doesn't match; `"no command in [project.scripts]"` if there are no scripts; `"unbounded dependency NAME"` for each dependency with none of `<>=!~` in it; and `"no [build-system]"` if that table is missing.

In [ ]:
import re
import tomllib

def pyproject_problems(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_waymark():\n    \"Waymark's file\"\n    assert pyproject_problems(open(\"pyproject.toml\").read()) == [\"unbounded dependency pandas\"]\n\ndef test_problems():\n    \"everything wrong\"\n    assert pyproject_problems(\"[project\\nname=\") == [\"not valid TOML\"]\n    assert pyproject_problems('[tool.x]\\na = 1\\n') == [\"no [project] table\"]\n    text = '[project]\\nname = \"waymark\"\\nversion = \"1.0\"\\ndependencies = [\"pandas>=2\", \"requests\"]\\n'\n    assert pyproject_problems(text) == [\"missing project.description\", \"missing project.readme\", \"missing project.requires-python\",\n                                        \"missing project.license\", \"version 1.0 is not MAJOR.MINOR.PATCH\",\n                                        \"no command in [project.scripts]\", \"unbounded dependency requests\", \"no [build-system]\"]")

### Exercise: Checksums

Write `checksums(files)`: `files` maps a file name to its bytes; return the text of a `SHA256SUMS` file, one line `DIGEST  NAME` (two spaces) per file sorted by name, ending with a newline. Then `verify(text, files)`: return a list of problems, in the order of the lines: `"NAME: missing"` if a listed file isn't in `files`, `"NAME: checksum mismatch"` if its digest differs.

In [ ]:
import hashlib

def checksums(files):
    ___

def verify(text, files):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import hashlib\n\nFILES = {\"waymark_walks-1.0.0.tar.gz\": b\"sdist bytes\", \"waymark_walks-1.0.0-py3-none-any.whl\": b\"wheel bytes\"}\n\ndef test_sums():\n    \"sha256sum's format\"\n    text = checksums(FILES)\n    lines = text.splitlines()\n    assert lines[0] == hashlib.sha256(b\"wheel bytes\").hexdigest() + \"  waymark_walks-1.0.0-py3-none-any.whl\"\n    assert lines[1].endswith(\"  waymark_walks-1.0.0.tar.gz\") and text.endswith(\"\\n\") and len(lines) == 2\n\ndef test_verify():\n    \"what you downloaded is what was built\"\n    text = checksums(FILES)\n    assert verify(text, FILES) == []\n    tampered = {\"waymark_walks-1.0.0.tar.gz\": b\"sdist bytes plus something\"}\n    assert verify(text, tampered) == [\"waymark_walks-1.0.0-py3-none-any.whl: missing\", \"waymark_walks-1.0.0.tar.gz: checksum mismatch\"]")

### Bug hunt: The latest release, by the alphabet

The release script finds the latest tag by comparing text, so it prints `v0.9.0`. Compare versions as numbers so it prints `v0.10.0`.

In [ ]:
tags = ["v0.9.0", "v0.10.0", "v0.2.1"]
print(max(tags))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"tuples of numbers\"\n    expect_output(\"v0.10.0\")")

### Bug hunt: A minor release that kept the patch

A new feature after `1.2.3` should be `1.3.0`, but this prints `1.3.3`. Reset what semver says to reset.

In [ ]:
def bump(version, part):
    major, minor, patch = (int(p) for p in version.split("."))
    if part == "major":
        major += 1
    elif part == "minor":
        minor += 1
    else:
        patch += 1
    return f"{major}.{minor}.{patch}"

print(bump("1.2.3", "minor"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"MINOR resets PATCH\"\n    expect_output(\"1.3.0\")\n    assert bump(\"1.2.3\", \"major\") == \"2.0.0\" and bump(\"1.2.3\", \"patch\") == \"1.2.4\"")

### Puzzle: The largest change wins

Arrange the loop that works out the level of release the commits ask for (3 major, 2 minor, 1 patch, 0 none). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # bump("1.4.2", "major")
q = ___   # the conventional commit type for a bug fix
r = ___   # the next version after 0.3.1 for the commit "feat!: new map format"
s = ___   # True or False: "1.10.0" > "1.9.0" as strings
t = ___   # the digest algorithm in a SHA256SUMS file (lower case, no hyphen)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"2.0.0\" and q == \"fix\" and r == \"0.4.0\", \"p, q and r\"\n    assert s is False and t == \"sha256\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: checking a release

Write `release_check(pyproject_text, changelog, tag, dist)`. `dist` maps file names to bytes and includes `SHA256SUMS` (as UTF-8 bytes). Run five checks and return `(lines, verdict)`, one line each:

1. `"pyproject: ok"`, or `"pyproject: "` and its problems (today's rules) joined by `"; "`;
2. `"tag: ok"` if `tag` is `"v"` plus the version, else `"tag: TAG doesn't match version VERSION"`;
3. `"changelog: ok"` if a line of `changelog` starts with `## [VERSION]`, else `"changelog: no entry for VERSION"`;
4. `"dist: ok"` if both `NAME-VERSION.tar.gz` and `NAME-VERSION-py3-none-any.whl` are present (NAME is the project name in lower case with `-` and `.` replaced by `_`), else `"dist: missing "` and the missing names (sdist first) joined by `", "`;
5. `"checksums: ok"` if `SHA256SUMS` exists, lists every other file in `dist`, and every listed file matches; else `"checksums: "` and the problems joined by `"; "`: `"no SHA256SUMS"`, or `"NAME not listed"` for unlisted files (sorted), then `"NAME: missing"` and `"NAME: checksum mismatch"` in the order of its lines.

`verdict` is `"RELEASE"` if every line ends in `ok`, otherwise `"STOP: "` and the names of the failing checks joined by `", "`.

In [ ]:
import hashlib
import re
import tomllib

def release_check(pyproject_text, changelog, tag, dist):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import hashlib\n\nPYPROJECT = open(\"pyproject.toml\").read().replace('\"pandas\"', '\"pandas>=2.1\"')\nCHANGELOG = \"# Changelog\\n\\n## [1.0.0] - 2026-11-16\\n\\n### Added\\n- everything\\n\"\n\ndef sums(files):\n    return \"\".join(f\"{hashlib.sha256(files[n]).hexdigest()}  {n}\\n\" for n in sorted(files)).encode()\n\ndef built():\n    files = {\"waymark_walks-1.0.0.tar.gz\": b\"sdist\", \"waymark_walks-1.0.0-py3-none-any.whl\": b\"wheel\"}\n    return {**files, \"SHA256SUMS\": sums(files)}\n\ndef test_release():\n    \"everything agrees\"\n    lines, verdict = release_check(PYPROJECT, CHANGELOG, \"v1.0.0\", built())\n    assert lines == [\"pyproject: ok\", \"tag: ok\", \"changelog: ok\", \"dist: ok\", \"checksums: ok\"] and verdict == \"RELEASE\"\n\ndef test_stop():\n    \"everything that can disagree\"\n    dist = built()\n    dist[\"waymark_walks-1.0.0.tar.gz\"] = b\"rebuilt\"\n    del dist[\"waymark_walks-1.0.0-py3-none-any.whl\"]\n    dist[\"notes.txt\"] = b\"x\"\n    lines, verdict = release_check(open(\"pyproject.toml\").read(), CHANGELOG.replace(\"1.0.0\", \"0.9.0\"), \"v1.0.1\", dist)\n    assert lines == [\n        \"pyproject: unbounded dependency pandas\",\n        \"tag: v1.0.1 doesn't match version 1.0.0\",\n        \"changelog: no entry for 1.0.0\",\n        \"dist: missing waymark_walks-1.0.0-py3-none-any.whl\",\n        \"checksums: notes.txt not listed; waymark_walks-1.0.0-py3-none-any.whl: missing; waymark_walks-1.0.0.tar.gz: checksum mismatch\",\n    ]\n    assert verdict == \"STOP: pyproject, tag, changelog, dist, checksums\"\n    del dist[\"SHA256SUMS\"]\n    assert release_check(PYPROJECT, CHANGELOG, \"v1.0.0\", dist)[0][-1] == \"checksums: no SHA256SUMS\"")

## Warm-up

As text, `"v0.9.0"` comes after `"v0.10.0"`, because `9` comes after `1`. Version numbers are tuples of numbers, and every tool that compares them has to treat them that way.

## Version numbers

**Semantic versioning** (semver.org, by Tom Preston-Werner) writes versions as `MAJOR.MINOR.PATCH` and gives each part a meaning:

- **PATCH** (1.0.0 to 1.0.1): bug fixes only; nothing anyone relies on changes.
- **MINOR** (1.0.1 to 1.1.0): new features that don't break anything; PATCH goes back to 0.
- **MAJOR** (1.1.0 to 2.0.0): something that worked before may not now (a renamed option, a removed API route); MINOR and PATCH go back to 0.

Before `1.0.0`, anything may change, and many projects bump the MINOR for breaking changes. Releasing `1.0.0` is a promise: from now on, a breaking change means a new MAJOR. Waymark's first public release is `1.0.0`.

## Commits that write the notes

If commit messages follow a pattern, the next version and the release notes can be worked out from them. **Conventional Commits** (conventionalcommits.org) starts each message with a type, an optional scope and a description:

- `feat(share): open a shared route as GeoJSON`: a new feature, so a MINOR release;
- `fix: round walks ignored the target time`: a fix, so a PATCH;
- `feat(api)!: rename /plan to /v1/routes`, or a body line `BREAKING CHANGE: ...`: a MAJOR;
- `docs:`, `test:`, `refactor:`, `ci:`, `chore:`: no release on their own.

Release notes are for users: group the entries under **Breaking**, **Added** and **Fixed**, in their words, and put them in `CHANGELOG.md` under the version and date (Week 90's changelog, now per release).

## The package

`pyproject.toml` (PEP 621) describes the package: its `name` (unique on the Python Package Index), `version`, a one-line `description`, the `readme`, `requires-python`, the `license`, its `dependencies` (with lower bounds, so an install doesn't pick a version that's too old), and `[project.scripts]`, which makes `waymark` a command when the package is installed. `[build-system]` says what builds it.

On your own computer, `python -m build` makes two files in `dist/`: a **source distribution** (`waymark_walks-1.0.0.tar.gz`) and a **wheel** (`waymark_walks-1.0.0-py3-none-any.whl`), with the name normalised to lower case and underscores. `twine upload --repository testpypi dist/*` publishes to TestPyPI first, so you can try `pip install` from there before the real thing. Both are free.

## Tags and checksums

Tag the commit that was released, `git tag -a v1.0.0 -m "Waymark 1.0.0"`, and push the tag; the tag, the version in `pyproject.toml` and the changelog heading must agree. Then publish a `SHA256SUMS` file: one line per file, the SHA-256 digest, two spaces and the file name (the format `sha256sum` writes and `sha256sum -c` checks). Anyone can then check that what they downloaded is what you built.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **Write conventional commits** from now on, and a `CHANGELOG.md` entry for your first release.
2. **Fill in `pyproject.toml`** properly and check it with `pyproject_problems`; for a web app, the equivalent is a pinned requirements file and a container or deployment description.
3. **Build and try it**: `python -m build`, install the wheel in a fresh virtual environment, run your command. Publish to TestPyPI if it's a package, or deploy to a free host if it's a service.
4. **Tag, checksum and release** on GitHub, with your release notes.

## Before you go

Tomorrow you'll write about what you built: a write-up that someone who has never seen your code can follow, with the decisions and the numbers behind them.